# 🎯 Embeddings y RAG: De la Teoría a la Práctica

**Curso:** Calidad de Software y Pruebas Automatizadas — Universidad de Nariño

**Módulo 1:** Requirements Refiner

**Objetivo:** Entender cómo la IA puede buscar conocimiento institucional para producir requerimientos de mayor calidad.

---

### ¿Qué vamos a hacer?

| Sección | Concepto | Pregunta que responde |
|---------|----------|----------------------|
| 1 | Tokenización | ¿Cómo ve un LLM el texto? |
| 2 | Embeddings | ¿Cómo se convierte texto en números con significado? |
| 3 | Similitud Coseno | ¿Cómo se mide si dos textos hablan de lo mismo? |
| 4 | ChromaDB | ¿Cómo se almacenan y buscan embeddings eficientemente? |
| 5 | RAG | ¿Cómo se usa todo esto para potenciar un LLM? |

Al final tendrás un mini-sistema RAG funcionando con historias de usuario reales del SGC de Katary.

---
## 📦 Instalación de dependencias

Ejecuta esta celda una sola vez. Instala las librerías que necesitamos.

In [ ]:
!pip install -q sentence-transformers chromadb
print("✅ Dependencias instaladas correctamente")

---
## Sección 1: Tokenización — ¿Cómo ve un LLM el texto?

Un LLM no lee letras ni palabras como nosotros. Lee **tokens**: fragmentos de texto que pueden ser palabras completas, partes de palabras, o incluso caracteres individuales.

Por ejemplo:
- `"autenticación"` podría ser 1 token o dividirse en `"aut"` + `"entic"` + `"ación"`
- `"login"` normalmente es 1 token porque es una palabra común en inglés

**¿Por qué importa?** Porque el tokenizador determina cómo el modelo "entiende" el texto. Un tokenizador entrenado con mucho español manejará mejor palabras como "autenticación" que uno entrenado solo con inglés.

In [ ]:
from transformers import AutoTokenizer

# Cargamos el tokenizador de un modelo de embeddings
tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

# Textos de ejemplo: requerimientos de software
textos = [
    "El sistema debe permitir login de usuarios",
    "Autenticación del sistema de gestión de proyectos",
    "El café está caliente",  # texto no relacionado para comparar
]

print("=" * 60)
print("TOKENIZACIÓN: cómo el modelo descompone cada texto")
print("=" * 60)

for texto in textos:
    tokens = tokenizer.tokenize(texto)
    ids = tokenizer.encode(texto, add_special_tokens=False)
    print(f"\n📝 Texto: \"{texto}\"")
    print(f"   Tokens ({len(tokens)}): {tokens}")
    print(f"   IDs:     {ids}")

### 🧠 Observa y reflexiona

- ¿Cuántos tokens tiene cada texto?
- ¿Las palabras en español se dividen en más tokens que las de inglés?
- ¿Por qué "login" probablemente es 1 token pero "autenticación" se divide?
- ¿Qué implicaciones tiene esto para un modelo que procesa requerimientos en español?

> **Concepto clave:** Un modelo entrenado con más texto en español necesitará menos tokens para representar el mismo texto, lo que significa que "entiende" mejor el idioma. Esto es relevante cuando elegimos modelos de embeddings.

---
## Sección 2: Embeddings — Texto convertido en números con significado

Un **embedding** es un vector (lista de números) que captura el **significado semántico** del texto.

Analogía: imagina que cada texto tiene coordenadas GPS en un espacio de significado. Textos que hablan de lo mismo están cerca. Textos sobre temas diferentes están lejos.

- `"login de usuario"` y `"autenticación del sistema"` → **coordenadas cercanas** (mismo tema)
- `"login de usuario"` y `"el café está caliente"` → **coordenadas lejanas** (temas diferentes)

La diferencia con los tokens es que los tokens son fragmentos sin contexto, mientras que el embedding captura el significado **completo** de la oración.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Cargamos un modelo de embeddings (80 MB, tarda ~30 seg la primera vez)
print("⏳ Cargando modelo all-MiniLM-L6-v2 (80 MB)...")
modelo = SentenceTransformer("all-MiniLM-L6-v2")
print("✅ Modelo cargado")

# Textos de ejemplo: requerimientos de software reales
textos = [
    "El sistema debe permitir login de usuarios con email y contraseña",
    "Autenticación de usuarios en el sistema de gestión de proyectos",
    "Registrar horas trabajadas por tarea con fecha y descripción",
    "Seguimiento del tiempo dedicado a las actividades del proyecto",
    "El café de la oficina está muy caliente hoy",
]

# Generar embeddings
embeddings = modelo.encode(textos)

print(f"\n📊 Resultados:")
print(f"   Cantidad de textos: {len(textos)}")
print(f"   Dimensiones por embedding: {embeddings.shape[1]}")
print(f"   Forma total: {embeddings.shape}")

# Veamos cómo se ve un embedding
print(f"\n🔍 Primeros 10 valores del embedding del texto 1:")
print(f"   {embeddings[0][:10]}")
print(f"   ... (384 dimensiones en total)")

### 🧠 Observa y reflexiona

- Cada texto se convierte en un vector de **384 números** (dimensiones)
- Esos 384 números codifican el **significado** del texto, no las palabras exactas
- Dos textos pueden usar palabras completamente diferentes pero tener embeddings similares si hablan del mismo tema

> **Pregunta:** ¿Cómo sabemos si dos embeddings son "similares"? → Eso lo resolvemos en la siguiente sección.

---
## Sección 3: Similitud Coseno — Midiendo cercanía semántica

La **similitud coseno** mide el ángulo entre dos vectores:

- **1.0** = vectores idénticos (misma dirección) → significado idéntico
- **0.0** = vectores perpendiculares → sin relación
- **-1.0** = vectores opuestos → significado opuesto (raro en la práctica)

En la práctica, para textos de requerimientos:
- **> 0.7** = muy similares (probablemente el mismo tema)
- **0.4 - 0.7** = algo relacionados
- **< 0.4** = no relacionados

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Calculamos similitud entre TODOS los pares de textos
similitudes = cosine_similarity(embeddings)

# Etiquetas cortas para la tabla
etiquetas = [
    "1. Login email",
    "2. Autenticación",
    "3. Registrar horas",
    "4. Seguim. tiempo",
    "5. Café caliente",
]

print("=" * 70)
print("MATRIZ DE SIMILITUD COSENO")
print("=" * 70)
print(f"{'':>20}", end="")
for i in range(len(etiquetas)):
    print(f"  [{i+1}]", end="")
print()

for i, etiq in enumerate(etiquetas):
    print(f"{etiq:>20}", end="")
    for j in range(len(etiquetas)):
        valor = similitudes[i][j]
        # Colorear: verde si > 0.7, amarillo si > 0.4
        if i == j:
            print(f" 1.00", end="")
        elif valor > 0.7:
            print(f" \033[92m{valor:.2f}\033[0m", end="")  # verde
        elif valor > 0.4:
            print(f" \033[93m{valor:.2f}\033[0m", end="")  # amarillo
        else:
            print(f" \033[91m{valor:.2f}\033[0m", end="")  # rojo
    print()

print("\n\033[92m■ Verde (> 0.7): muy similares\033[0m")
print("\033[93m■ Amarillo (0.4-0.7): algo relacionados\033[0m")
print("\033[91m■ Rojo (< 0.4): no relacionados\033[0m")

### 🧠 Observa y reflexiona

Deberías ver algo como:
- **Textos 1 y 2** (login ↔ autenticación): similitud **alta** (~0.6-0.8) → mismo tema, diferentes palabras
- **Textos 3 y 4** (registrar horas ↔ seguimiento tiempo): similitud **alta** (~0.6-0.8) → mismo dominio
- **Texto 5** (café): similitud **baja** con todos (~0.1-0.3) → tema no relacionado

> **Momento \"aha\":** El modelo entiende que \"login\" y \"autenticación\" hablan de lo mismo **sin que nadie se lo haya dicho explícitamente**. Lo aprendió leyendo millones de textos durante su entrenamiento.

### 🧪 Ejercicio práctico: experimenta tú

Modifica los textos de abajo con requerimientos de tu propia experiencia y observa las similitudes.

In [ ]:
# === MODIFICA ESTOS TEXTOS CON TUS PROPIOS REQUERIMIENTOS ===
mis_textos = [
    "Gestionar usuarios del sistema con roles y permisos",
    "Administración de cuentas de usuario con control de acceso",
    "Generar reporte PDF con gráficos de avance del proyecto",
    # Agrega más textos aquí...
]

# Generar embeddings y calcular similitudes
mis_embeddings = modelo.encode(mis_textos)
mis_similitudes = cosine_similarity(mis_embeddings)

print("Similitudes entre tus textos:\n")
for i in range(len(mis_textos)):
    for j in range(i + 1, len(mis_textos)):
        sim = mis_similitudes[i][j]
        emoji = "🟢" if sim > 0.7 else "🟡" if sim > 0.4 else "🔴"
        print(f"{emoji} {sim:.3f}  \"{mis_textos[i][:50]}\"")
        print(f"         ↔ \"{mis_textos[j][:50]}\"\n")

---
## Sección 4: ChromaDB — Base de datos vectorial

Ya sabemos convertir texto en vectores y medir similitud. Pero en un proyecto real tenemos **cientos de historias de usuario** en la base de conocimiento. No podemos calcular similitud contra todas cada vez.

**ChromaDB** es una base de datos vectorial que:
1. **Almacena** embeddings con sus metadatos
2. **Indexa** los vectores para búsqueda rápida
3. **Busca** los K vectores más cercanos a una consulta (K-Nearest Neighbors)

Es como un Google para significado: en vez de buscar por palabras clave, busca por **cercanía semántica**.

Vamos a cargar historias de usuario reales del SGC de Katary Software y buscar en ellas.

In [ ]:
import chromadb

# Base de conocimiento: historias de usuario del SGC de Katary (Katary360)
# Estas son historias reales con nivel de calidad CMMI-DEV Nivel 3
katary_stories = [
    {
        "id": "SGC-US-001",
        "texto": "Como líder de proyecto, quiero crear un nuevo proyecto ingresando nombre, cliente, fecha de inicio y fecha fin estimada, para que el equipo pueda comenzar a planificar las actividades.",
        "criterios": "GIVEN líder autenticado WHEN ingresa nombre (5-100 chars), selecciona cliente, fecha inicio (no anterior a hoy), fecha fin (posterior a inicio) THEN crea proyecto con estado Planificación, genera código PRY-YYYYMMDD-NNN en < 2 seg",
        "dominio": "gestion_proyectos"
    },
    {
        "id": "SGC-US-002",
        "texto": "Como desarrollador, quiero registrar las horas trabajadas por tarea indicando fecha, cantidad de horas y descripción, para que el líder pueda hacer seguimiento del esfuerzo real vs estimado.",
        "criterios": "GIVEN desarrollador asignado a tarea activa WHEN ingresa fecha (no futura, max 7 días atrás), horas (0.5-16, incrementos 0.5), descripción (min 20 chars) THEN registra tiempo, actualiza total y recalcula avance",
        "dominio": "seguimiento_tiempo"
    },
    {
        "id": "SGC-US-003",
        "texto": "Como administrador, quiero gestionar usuarios del sistema pudiendo crear, editar, desactivar y reactivar cuentas con roles específicos, para que solo personal autorizado acceda a las funcionalidades.",
        "criterios": "GIVEN admin autenticado WHEN crea usuario con email @katary.co, nombre completo, roles (admin/líder/dev/QA/cliente) THEN crea cuenta activa, envía email con contraseña temporal (expira 24h), registra en log auditoría",
        "dominio": "autenticacion"
    },
    {
        "id": "SGC-US-006",
        "texto": "Como usuario, quiero iniciar sesión con email y contraseña, y que el sistema bloquee mi cuenta tras 5 intentos fallidos, para proteger el acceso no autorizado.",
        "criterios": "GIVEN usuario con cuenta activa WHEN ingresa credenciales correctas THEN inicia sesión, registra fecha/hora/IP en log, redirige a dashboard en < 2 seg. GIVEN 4 intentos fallidos WHEN falla el 5to THEN bloquea cuenta 30 min, envía email al usuario y admin",
        "dominio": "autenticacion"
    },
    {
        "id": "SGC-US-005",
        "texto": "Como gerente, quiero ver un dashboard con indicadores clave de todos los proyectos activos incluyendo avance, semáforo, horas consumidas vs presupuestadas y tareas pendientes.",
        "criterios": "GIVEN gerente autenticado WHEN accede al dashboard THEN muestra en < 3 seg: proyectos con semáforo (verde >= plan, amarillo < 10% retraso, rojo >= 10%), gráfico horas consumidas vs presupuestadas, tabla tareas por estado",
        "dominio": "reportes"
    },
    {
        "id": "SGC-US-008",
        "texto": "Como analista de calidad, quiero registrar defectos encontrados durante las pruebas especificando severidad, componente, pasos para reproducir y evidencia.",
        "criterios": "GIVEN analista QA asignado WHEN registra defecto con título (10-200 chars), severidad (bloqueante/crítico/mayor/menor/trivial), componente, min 3 pasos reproducir, resultado esperado vs obtenido, min 1 captura THEN crea defecto estado nuevo, vincula a caso de prueba, notifica al dev",
        "dominio": "calidad"
    },
    {
        "id": "SGC-US-013",
        "texto": "Como analista de calidad, quiero generar una matriz de trazabilidad que vincule cada requerimiento con sus historias, casos de prueba y defectos asociados.",
        "criterios": "GIVEN proyecto con min 3 requerimientos, historias y casos de prueba WHEN solicita matriz THEN genera en < 5 seg tabla con: ID req, descripción, historias, casos (total/pasados/fallidos), defectos abiertos, cobertura %. Sin cobertura = amarillo, sin casos = rojo",
        "dominio": "calidad"
    },
    {
        "id": "SGC-US-014",
        "texto": "Como usuario, quiero recuperar mi contraseña mediante un enlace enviado a mi email corporativo, para acceder nuevamente sin depender del administrador.",
        "criterios": "GIVEN usuario con cuenta activa WHEN solicita recuperación ingresando email THEN envía enlace (válido 1 hora, uso único), mensaje genérico sin revelar si email existe. GIVEN enlace válido WHEN establece nueva contraseña (min 8 chars, 1 mayúscula, 1 minúscula, 1 número, 1 especial, diferente a últimas 5) THEN actualiza, invalida sesiones, confirma por email",
        "dominio": "autenticacion"
    },
    {
        "id": "SGC-US-015",
        "texto": "Como líder de proyecto, quiero registrar y hacer seguimiento de riesgos indicando descripción, probabilidad, impacto, estrategia de mitigación y responsable.",
        "criterios": "GIVEN líder de proyecto activo WHEN registra riesgo con descripción (20-500 chars), probabilidad (alta/media/baja), impacto (alto/medio/bajo), categoría, estrategia mitigación (20-1000 chars), responsable THEN calcula nivel (prob x impacto), agrega a matriz, programa revisión quincenal",
        "dominio": "gestion_riesgos"
    }
]

print(f"📚 Base de conocimiento cargada: {len(katary_stories)} historias del SGC de Katary")
print(f"   Dominios: {set(s['dominio'] for s in katary_stories)}")

In [ ]:
# Crear base de datos vectorial en memoria
client = chromadb.Client()  # En memoria (Colab)

# Crear una colección (equivalente a una "tabla" en SQL)
collection = client.create_collection(
    name="katary_sgc",
    metadata={"hnsw:space": "cosine"}  # Usar similitud coseno
)

# Insertar historias en ChromaDB
# ChromaDB genera los embeddings automáticamente con su modelo por defecto,
# pero nosotros vamos a usar nuestro propio modelo para tener control total

textos_historias = [s["texto"] for s in katary_stories]
embeddings_historias = modelo.encode(textos_historias).tolist()

collection.add(
    ids=[s["id"] for s in katary_stories],
    embeddings=embeddings_historias,
    documents=textos_historias,
    metadatas=[{"dominio": s["dominio"], "criterios": s["criterios"]} for s in katary_stories],
)

print(f"✅ {collection.count()} historias indexadas en ChromaDB")
print(f"   Cada historia almacenada con: embedding (384 dim) + texto + metadatos")

In [ ]:
# === BÚSQUEDA SEMÁNTICA: encontrar historias similares ===

def buscar_historias(consulta: str, top_k: int = 3):
    """Busca las historias más similares a la consulta."""
    # Generar embedding de la consulta
    query_embedding = modelo.encode([consulta]).tolist()

    # Buscar en ChromaDB
    resultados = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )

    print(f"\n🔍 Consulta: \"{consulta}\"")
    print(f"{'=' * 60}")

    for i in range(len(resultados['ids'][0])):
        story_id = resultados['ids'][0][i]
        documento = resultados['documents'][0][i]
        metadata = resultados['metadatas'][0][i]
        # ChromaDB con cosine devuelve distancia (1 - similitud)
        distancia = resultados['distances'][0][i]
        similitud = 1 - distancia

        emoji = "🟢" if similitud > 0.5 else "🟡" if similitud > 0.3 else "🔴"
        print(f"\n{emoji} #{i+1} [{story_id}] Similitud: {similitud:.3f}")
        print(f"   Dominio: {metadata['dominio']}")
        print(f"   Historia: {documento[:120]}...")
        print(f"   Criterios: {metadata['criterios'][:120]}...")

    return resultados


# Probemos con un requerimiento nuevo
buscar_historias("Necesito un sistema de login seguro")

In [ ]:
# Probemos con más consultas para ver cómo la búsqueda semántica encuentra
# historias relevantes aunque las palabras sean diferentes

buscar_historias("Quiero que los empleados registren cuánto tiempo trabajan cada día")

print("\n" + "#" * 60 + "\n")

buscar_historias("Necesito ver cómo van todos los proyectos de la empresa")

print("\n" + "#" * 60 + "\n")

buscar_historias("Quiero reportar bugs que encuentro cuando pruebo el software")

### 🧠 Observa y reflexiona

- La búsqueda encontró historias relevantes **aunque las palabras sean completamente diferentes**
- `"reportar bugs"` encuentra la historia de `"registrar defectos"` → entiende el significado, no las palabras
- `"cuánto tiempo trabajan"` encuentra `"registrar horas trabajadas"` → mismo concepto, diferente expresión
- Los criterios Given/When/Then de las historias encontradas son **ejemplos de calidad CMMI** que el agente puede usar como referencia

> **Esto es la base del RAG:** buscar conocimiento previo relevante antes de generar contenido nuevo.

### 🧪 Ejercicio: busca con tus propios requerimientos

In [ ]:
# === ESCRIBE TU PROPIO REQUERIMIENTO Y BUSCA EN LA KB ===
mi_requerimiento = "Escribe aquí tu requerimiento"

buscar_historias(mi_requerimiento, top_k=3)

---
## Sección 5: RAG — Retrieval-Augmented Generation

Ahora juntamos todo. **RAG** es el patrón que hace que un LLM genérico se convierta en un asistente especializado:

```
1. Usuario escribe requerimiento en texto libre
2. Sistema busca historias similares en la KB (lo que acabamos de hacer)
3. Las historias encontradas se inyectan en el prompt del LLM
4. El LLM genera con contexto institucional, no genéricamente
```

La diferencia es brutal:
- **Sin RAG:** el LLM inventa criterios genéricos basados en su entrenamiento general
- **Con RAG:** el LLM ve ejemplos reales del SGC de Katary y produce criterios del mismo nivel de calidad

Vamos a simularlo completo. No necesitamos un LLM real para ver cómo se construye el prompt enriquecido.

In [ ]:
def construir_prompt_rag(requerimiento: str, top_k: int = 3) -> str:
    """Construye el prompt enriquecido con contexto de la KB.

    Este es el corazón del RAG: buscar conocimiento relevante
    e inyectarlo en el prompt antes de llamar al LLM.
    """
    # Paso 1: Buscar historias similares
    query_embedding = modelo.encode([requerimiento]).tolist()
    resultados = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )

    # Paso 2: Construir el contexto con las historias encontradas
    contexto_kb = "## HISTORIAS DE REFERENCIA DEL SGC DE KATARY\n"
    contexto_kb += "Usa estas historias como modelo de calidad y profundidad:\n\n"

    for i in range(len(resultados['ids'][0])):
        story_id = resultados['ids'][0][i]
        documento = resultados['documents'][0][i]
        metadata = resultados['metadatas'][0][i]
        similitud = 1 - resultados['distances'][0][i]

        contexto_kb += f"### Referencia {i+1} [{story_id}] (similitud: {similitud:.2f})\n"
        contexto_kb += f"**Historia:** {documento}\n"
        contexto_kb += f"**Criterios:** {metadata['criterios']}\n"
        contexto_kb += f"**Dominio:** {metadata['dominio']}\n\n"

    # Paso 3: Construir el prompt completo para el LLM
    prompt = f"""Eres un Analista de Requerimientos Senior de Katary Software.
Tu trabajo es transformar requerimientos ambiguos en historias de usuario
con criterios de aceptación Given/When/Then verificables.

{contexto_kb}

## REQUERIMIENTO A ANALIZAR:
{requerimiento}

Genera historias de usuario con el mismo nivel de calidad y detalle
que las historias de referencia del SGC de Katary."""

    return prompt, resultados


# === DEMO: ver cómo se ve un prompt enriquecido con RAG ===
requerimiento = "Necesito un sistema de login seguro para la plataforma"

prompt, resultados = construir_prompt_rag(requerimiento)

print("=" * 70)
print("PROMPT ENRIQUECIDO CON RAG")
print("Este es lo que recibiría el LLM (Gemma/Claude/GPT)")
print("=" * 70)
print(prompt)

In [ ]:
# === COMPARACIÓN: prompt SIN RAG vs CON RAG ===

prompt_sin_rag = f"""Eres un Analista de Requerimientos.
Transforma este requerimiento en historias de usuario con criterios
de aceptación Given/When/Then.

REQUERIMIENTO:
{requerimiento}"""

print("=" * 70)
print("❌ PROMPT SIN RAG (genérico)")
print("=" * 70)
print(prompt_sin_rag)
print(f"\nLongitud: {len(prompt_sin_rag)} caracteres")
print("\nEl LLM solo tiene instrucciones generales.")
print("No sabe cómo Katary escribe criterios, qué nivel de detalle espera,")
print("ni qué patrones usar para autenticación.")

print(f"\n\n{'=' * 70}")
print("✅ PROMPT CON RAG (informado por KB de Katary)")
print("=" * 70)
print(f"Longitud: {len(prompt)} caracteres")
print(f"\nEl LLM recibe {len(resultados['ids'][0])} historias de referencia reales.")
print("Sabe exactamente:")
print("  - Cómo Katary estructura criterios Given/When/Then")
print("  - Qué nivel de detalle se espera (tiempos, validaciones, mensajes)")
print("  - Qué patrones usa Katary para autenticación (bloqueo 5 intentos, etc.)")
print(f"\nDiferencia: {len(prompt) - len(prompt_sin_rag)} caracteres de contexto institucional")

### 🧠 Reflexión final

Acabas de construir un sistema RAG funcional. Recapitulemos:

1. **Tokenización**: el texto se descompone en fragmentos que el modelo puede procesar
2. **Embeddings**: cada texto se convierte en un vector de 384 números que captura su significado
3. **Similitud coseno**: medimos qué tan parecidos son dos textos por la dirección de sus vectores
4. **ChromaDB**: almacena y busca embeddings eficientemente entre cientos de documentos
5. **RAG**: busca conocimiento relevante en la KB y lo inyecta en el prompt del LLM

### ¿Por qué esto importa para la calidad del software?

- **Sin RAG**: el LLM produce historias genéricas → calidad impredecible
- **Con RAG**: el LLM produce historias informadas por 19 años de experiencia de Katary → calidad CMMI-DEV L3

La base de conocimiento es la **memoria institucional** de la organización. El RAG es el mecanismo que la hace accesible al agente de IA.

### Próximo paso

En el trabajo autónomo, instalarás Ollama + Gemma 4 en tu máquina local y conectarás este sistema RAG con un LLM real para ver las historias de usuario generadas con contexto institucional.

---
## 🎯 Ejercicio de trabajo autónomo

### Parte A: Experimenta con modelos

Cambia el modelo de embeddings y compara resultados:

```python
# Modelo rápido (el que usamos)
modelo_rapido = SentenceTransformer("all-MiniLM-L6-v2")  # 80 MB, 384 dim

# Modelo multilingual (mejor para español)
modelo_es = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")  # 471 MB, 384 dim
```

Preguntas para explorar:
- ¿El modelo multilingual encuentra mejores resultados para textos en español?
- ¿Cambian las similitudes significativamente?
- ¿Cuánto más tarda en generar embeddings?

### Parte B: Amplía la base de conocimiento

Agrega 5 historias de usuario de tu propia experiencia laboral o académica a `katary_stories` y reindexalas en ChromaDB. Luego prueba búsquedas que deberían encontrar tus historias nuevas.

### Parte C: Comparación de modelos (avanzado)

En la celda de abajo, implementa una función que compare los resultados de búsqueda entre el modelo básico y el multilingual para la misma consulta.

In [ ]:
# Espacio para tu experimentación

# Parte A: Carga un modelo diferente
# modelo_es = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# Parte B: Agrega historias
# mis_historias = [
#     {"id": "MI-US-001", "texto": "...", "criterios": "GIVEN... WHEN... THEN...", "dominio": "..."},
# ]

# Parte C: Función de comparación
# def comparar_modelos(consulta, modelo1, modelo2, collection):
#     pass